### Installation

In [1]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token_test")


# Login ke Hugging Face
login(token=hf_token)

In [3]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_secret = user_secrets.get_secret("wandb")
# Ganti dengan API Key rahasia Anda
wandb.login(key=wandb_secret)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jauharulumam (jauharulumam-uin-walisongo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
wandb.init(
    project="finetune-jdih-kemenkes", # Nama folder proyek di dashboard W&B
    name="percobaan-1-lora"           # Nama eksperimen spesifik ini
)

### Unsloth

In [5]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-0.6B-unsloth-bnb-4bit"
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 6.008 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 9.804 GiB requested
  cuda:0  budget  12.98 GiB  weights  3.141 GiB  free  9.837 GiB  reserve  9.804 GiB
  cuda:1  budget  13.00 GiB  weights  2.866 GiB  free 10.130 GiB  reserve  9.634 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_mem

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2026.9.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [7]:
from datasets import load_dataset

# Ganti nama file sesuai dengan file yang Anda simpan sebelumnya
# Bisa berupa .json atau .jsonl
nama_file = "/kaggle/input/datasets/jauharulumam/qa-augmented/QA_augmented_data.json" 

# Memuat data ke dalam format DatasetDict
dataset_loaded = load_dataset("json", data_files=nama_file)

# load_dataset secara default mengembalikan struktur "DatasetDict"
# Kita perlu mengambil split "train" agar kembali menjadi objek "Dataset" biasa
my_dataset = dataset_loaded["train"]

# Cek hasil load
print(my_dataset)
print(f"Jumlah baris: {my_dataset.num_rows}")
print("\nContoh data pertama:")
print(my_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['conversations'],
    num_rows: 776
})
Jumlah baris: 776

Contoh data pertama:
{'conversations': [{'role': 'user', 'content': 'Sebutkan bunyi Pasal 1.'}, {'role': 'assistant', 'content': 'Pasal 1\n\nDalam Peraturan Menteri ini yang dimaksud dengan: 1. Jaringan Dokumentasi dan Informasi Hukum Nasional yang selanjutnya disingkat JDIHN adalah wadah pendayagunaan bersama atas dokumen hukum secara tertib, terpadu, dan berkesinambungan, serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah, dan cepat. 2. Jaringan Dokumentasi dan Informasi Hukum di lingkungan Kementerian Kesehatan yang selanjutnya disebut JDIH Kemenkes adalah suatu sistem pengelolaan dan pendayagunaan bersama dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpadu dan berkesinambungan serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah dan cepat. 3. Dokumen Hukum adalah produk hukum yang berupa peraturan perun

Next we take the non reasoning dataset and convert it to conversational format as well.

We have to use Unsloth's `standardize_sharegpt` function to fix up the format of the dataset first.

In [10]:
from unsloth.chat_templates import standardize_sharegpt

# 1. Standarisasi format percakapan
dataset = standardize_sharegpt(my_dataset)

# 2. Terapkan chat template (mengubah format chat menjadi satu string teks per baris)
formatted_conversations = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
)

# 3. Langsung buat Dataset final dengan kolom "text"
final_dataset = Dataset.from_dict({"text": formatted_conversations})

# 4. Acak urutan data
final_dataset = final_dataset.shuffle(seed=3407)

print(final_dataset)

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/776 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 776
})


In [11]:
final_dataset[20]

{'text': '<|im_start|>user\nKepada pihak mana Pusat JDIH Kemenkes menyampaikan laporan mengenai penyelenggaraan JDIH Kemenkes, dan melalui siapa laporan yang ditujukan kepada Menteri disampaikan?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nLaporan tersebut disampaikan kepada Menteri melalui Sekretaris Jenderal serta kepada Pusat JDIHN. Ketentuan mengenai penerima dan mekanisme penyampaian laporan ini diatur dalam Pasal 4 ayat (2) huruf h.<|im_end|>\n'}

In [12]:
from datasets import DatasetDict

# 1. Split pertama: Ambil 20% untuk gabungan Eval dan Test (test_size=0.2)
initial_split = final_dataset.train_test_split(test_size=0.3, seed=3407)

train_split = initial_split["train"] # Ini adalah 80% data asli
eval_test_combined = initial_split["test"] # Ini adalah 20% sisa data

# 2. Split kedua: Bagi yang 20% tadi menjadi 2 bagian yang sama rata (test_size=0.5)
second_split = eval_test_combined.train_test_split(test_size=0.5, seed=3407)

eval_split = second_split["train"] # Setengah dari 20% = 10% dari total awal
test_split = second_split["test"]  # Setengah dari 20% = 10% dari total awal

# 3. (Opsional) Gabungkan kembali ke dalam satu objek DatasetDict agar rapi
final_split_dataset = DatasetDict({
    "train": train_split,
    "eval": eval_split,
    "test": test_split
})

# Cek hasil jumlah baris masing-masing
print(final_split_dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 543
    })
    eval: Dataset({
        features: ['text'],
        num_rows: 116
    })
    test: Dataset({
        features: ['text'],
        num_rows: 117
    })
})


In [13]:
final_split_dataset['train']

Dataset({
    features: ['text'],
    num_rows: 543
})

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [14]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = final_split_dataset['train'],       # Gunakan pecahan data latih
    eval_dataset = final_split_dataset['eval'],         # Gunakan pecahan data evaluasi
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4, 
        warmup_steps = 20,
        num_train_epochs = 7, 
        learning_rate = 2e-4, 
        logging_steps = 1,
        
        # --- KONFIGURASI EVALUASI ---
        eval_strategy = "steps",        # Lakukan evaluasi berbasis langkah (bisa juga "epoch")
        eval_steps = 15,                 # Lakukan evaluasi setiap 5 langkah
        per_device_eval_batch_size = 2, # Ukuran batch saat proses evaluasi berjalan
        # ----------------------------
        save_strategy = "epoch",        
        save_total_limit = 3,           
        output_dir = "./outputs",       
        # -----------------------------------------

        # --- KONFIGURASI AUTO-SAVE KE HUGGING FACE ---
        push_to_hub = True,                                 # Aktifkan upload otomatis
        hub_model_id = "Jauharul/qwen3-8b-lora-permenkes-7epoch",  # GANTI dengan username HF & nama repo Anda
        hub_strategy = "every_save",                        # Upload setiap kali save_strategy (tiap epoch) terpicu
        hub_private_repo = False,                            # (Opsional) Ubah ke False jika ingin modelnya publik
        # ---------------------------------------------

        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "wandb", 
        padding_free = False, 
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/543 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/116 [00:00<?, ? examples/s]

In [15]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.562 GB.
4.283 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [16]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 543 | Num Epochs = 7 | Total steps = 238
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
15,1.377200,1.325873
30,0.796300,0.802235
45,0.585200,0.615320
60,0.378200,0.530059
75,0.290800,0.471880
90,0.338400,0.441297
105,0.216300,0.434215
120,0.233800,0.438291
135,0.233200,0.423695
150,0.200100,0.494260


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [17]:
wandb.finish()

eval/loss,█▄▂▂▁▁▁▁▁▂▁▂▂▂▂
eval/runtime,▅▁▁▂▄▃▂▄▂▂▄▂▁█▂
eval/samples_per_second,▄██▇▅▆▇▅▇▇▅▇█▁▇
eval/steps_per_second,▄██▇▅▆▇▅▇▇▅▇█▁▇
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇████
train/grad_norm,█▅▃▂▂▃▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▄▃▂▂▂▇▂▃▃▂▂▂▂▁▁
train/learning_rate,▂▂▅▆█▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁
train/loss,█▄▄▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁
eval/loss,0.56743
eval/runtime,24.8752


In [18]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

2883.7813 seconds used for training.
48.06 minutes used for training.
Peak reserved memory = 6.465 GB.
Peak reserved memory for training = 2.182 GB.
Peak reserved memory % of max memory = 44.396 %.
Peak reserved memory for training % of max memory = 14.984 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for reasoning inference are `temperature = 0.6, top_p = 0.95, top_k = 20`

For normal chat based inference, `temperature = 0.7, top_p = 0.8, top_k = 20`

In [23]:
messages = [
    {"role" : "user", "content" : "Sebutkan bunyi pasal 4"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1024, # Increase for longer outputs!
    temperature = 0.2, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Pasal 4

(1) Pusat JDIH Kemenkes melaksanakan monitoring dan evaluasi terhadap:

a. pelaksanaan pengelolaan JDIH Kemenkes; dan

b. pelaksanaan tugas dan fungsi anggota JDIH Kemenkes.

(2) Monitoring dan evaluasi sebagaimana dimaksud pada ayat (1) dilaksanakan paling sedikit 1 (satu) kali dalam setiap tahun.<|im_end|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [24]:
"""
# model.save_pretrained("qwen_lora")  # Local saving
# tokenizer.save_pretrained("qwen_lora")
model.push_to_hub("Jauharul/qwen3-0.8b_lora16_permenkes", token = hf_token) # Online saving
tokenizer.push_to_hub("Jauharul/qwen3-0.8b_lora16_permenkes", token = hf_token) # Online saving
"""

README.md:   0%|          | 0.00/560 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Jauharul/qwen3-0.8b_lora16_permenkes


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [25]:
if True:
    from unsloth import FastLanguageModel
    model2, tokenizer2 = FastLanguageModel.from_pretrained(
        model_name = "Jauharul/qwen3-8b-lora-permenkes-7epoch", # YOUR MODEL YOU USED FOR TRAINING
        revision = "61d47d10cd6472db02f9913893877c266ecff3dd",
        max_seq_length = 2048,
        load_in_4bit = True,
    )

==((====))==  Unsloth 2026.9.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 6.008 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 6.030 GiB requested
  cuda:0  budget   8.67 GiB  weights  2.782 GiB  free  5.885 GiB  reserve  5.841 GiB
  cuda:1  budget   9.76 GiB  weights  3.226 GiB  free  6.534 GiB  reserve  6.030 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 9.630 -> 8.667 GiB, cuda:1 10.845 -> 9.760 GiB (memory the quantiser keeps for its own load-time buffers)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [27]:
messages = [
    {"role" : "user", "content" : "Apa itu JDIHN"}
]
text = tokenizer2.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

from transformers import TextStreamer
_ = model2.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.2, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
    repetition_penalty = 1.15,
)

JDIHN adalah singkatan dari Jaringan Dokumentasi dan Informasi Hukum Nasional. Menurut Pasal 1 angka 1, JDIHN merupakan wadah pendayagunaan bersama dokumen hukum secara tertib, terpadu, dan berkesinambungan; serta sarana pelayanan informasi hukum yang lengkap, akurat, mudah diakses, dan cepat.<|im_end|>


In [40]:
from unsloth import FastLanguageModel
import pandas as pd

# 1. Aktifkan mode inferensi
FastLanguageModel.for_inference(model2)

hasil_prediksi = []

print(f"Memulai inferensi untuk {len(final_split_dataset['test'])} data test...\n")

for i, row in enumerate(final_split_dataset['test']):
    full_text = row["text"]
    
    # 2. Memisahkan Prompt dan Jawaban Asli
    parts = full_text.split("<|im_start|>assistant\n")
    prompt_text = parts[0] + "<|im_start|>assistant\n"
    jawaban_asli = parts[1].replace("<|im_end|>\n", "").strip() if len(parts) > 1 else ""
    
    # 3. MENGAMBIL USER QUERY (PERTANYAAN)
    # Kita ambil teks setelah tag 'user', lalu potong tepat sebelum tag 'im_end'
    pertanyaan = ""
    if "<|im_start|>user\n" in prompt_text:
        pertanyaan = prompt_text.split("<|im_start|>user\n")[-1].split("<|im_end|>")[0].strip()
    
    # 4. Tokenisasi teks
    inputs = tokenizer2(prompt_text, return_tensors="pt").to("cuda")
    
    # 5. Generate jawaban dari model
    outputs = model2.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=1024,
        use_cache=True,
        do_sample=False
    )
    
    # 6. Potong output agar hanya mengambil jawaban model
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    jawaban_model = tokenizer2.decode(generated_ids, skip_special_tokens=True)
    
    # 7. Simpan hasil (Sekarang pertanyaan sudah ikut!)
    hasil_prediksi.append({
        "pertanyaan": pertanyaan,
        "jawaban_asli": jawaban_asli,
        "jawaban_model": jawaban_model
    })
    
    if (i + 1) % 10 == 0:
        print(f"Selesai memproses {i + 1} data...")

print("\nInferensi Selesai!")

# --- SIMPAN KE JSON ---
nama_file_json = "hasil_evaluasi_model_lengkap.json"

with open(nama_file_json, "w", encoding="utf-8") as f:
    json.dump(hasil_prediksi, f, ensure_ascii=False, indent=4)

print(f"File berhasil disimpan ke: {nama_file_json}")

Memulai inferensi untuk 117 data test...

Selesai memproses 10 data...
Selesai memproses 20 data...
Selesai memproses 30 data...
Selesai memproses 40 data...
Selesai memproses 50 data...
Selesai memproses 60 data...
Selesai memproses 70 data...
Selesai memproses 80 data...
Selesai memproses 90 data...
Selesai memproses 100 data...
Selesai memproses 110 data...

Inferensi Selesai!
File berhasil disimpan ke: hasil_evaluasi_model_lengkap.json


In [42]:
hasil_prediksi[0:5]

[{'pertanyaan': 'Mohon sebutkan jenis Dokumen Hukum yang termasuk dalam sistem pengelolaan JDIH Kemenkes.',
  'jawaban_asli': '<think>\n\n</think>\n\nJenis Dokumen Hukum yang termasuk dalam pengelolaan JDIH Kemenkes adalah Peraturan Perundang-undangan, produk hukum lain, monografi, artikel hukum, dan putusan/yurisprudensi, sebagaimana dinyatakan dalam Pasal 8 ayat (1).',
  'jawaban_model': '<think>\n\n</think>\n\nJenis Dokumen Hukum yang dikelola meliputi Peraturan Perundang-undangan, produk hukum lain, monografi, artikel hukum, dan putusan/yurisprudensi. Dasar pengelolaan ini tercantum dalam Pasal 8 ayat (3).'},
 {'pertanyaan': 'Apa isi atau bunyi Pasal 7?',
  'jawaban_asli': '<think>\n\n</think>\n\nPasal 7\n\n(1) Sebagai pendukung pengelolaan dokumentasi dan Informasi Hukum sebagaimana diatur dalam Pasal 6, dibentuk tim teknis JDIH Kemenkes.\n\n(2) Tim teknis JDIH Kemenkes sebagaimana dimaksud pada ayat (1) dibentuk dari unsur pusat JDIH Kemenkes, anggota JDIH Kemenkes, dan Pusat Dat